In [1]:
import os
import re
import glob
import zipfile
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import cv2
import mediapipe as mp
from tqdm.auto import tqdm
import joblib

print(f"NumPy version:     {np.__version__}")
print(f"OpenCV version:    {cv2.__version__}")
print(f"MediaPipe version: {mp.__version__}")
assert hasattr(mp, "solutions"), "ERROR: mp.solutions not found! Ensure mediapipe==0.10.14 is installed."
print("SUCCESS: mp.solutions.holistic is ready!")

NumPy version:     1.26.4
OpenCV version:    4.11.0
MediaPipe version: 0.10.14
SUCCESS: mp.solutions.holistic is ready!


In [2]:
# Locate Downloads folder automatically
DOWNLOADS_DIR = Path.home() / "Downloads"
ZIP_FILES = [
    DOWNLOADS_DIR / "Adjectives_1of8.zip",
    DOWNLOADS_DIR / "Greetings_1of2.zip"
]

DATA_DIR = Path("dataset_extracted")
DATA_DIR.mkdir(exist_ok=True)

for zpath in ZIP_FILES:
    if not zpath.exists():
        raise FileNotFoundError(f"Could not find {zpath}. Check the path!")
    print(f"Checking/Extracting {zpath.name}...")
    with zipfile.ZipFile(zpath, 'r') as zf:
        for member in zf.namelist():
            # Skip macOS metadata artifacts
            if "__MACOSX" in member or Path(member).name.startswith("._"):
                continue
            target_path = DATA_DIR / member
            if not target_path.exists():
                zf.extract(member, DATA_DIR)

# Target 13 ISL classes
TARGET_WORDS = [
    "loud", "quiet", "happy", "sad", "beautiful", "ugly", "deaf", "blind",
    "hello", "how are you", "alright", "good morning", "good afternoon"
]

def clean_folder_label(folder_name: str) -> str:
    # Strip leading numbers like '49. How are you' -> 'how are you'
    cleaned = re.sub(r'^\d+\.\s*', '', folder_name.strip())
    cleaned = re.sub(r'\s+', ' ', cleaned).lower()
    return cleaned

video_records = []
unmatched_folders = set()

for root, dirs, files in os.walk(DATA_DIR):
    mov_files = [
        f for f in files 
        if f.lower().endswith(('.mov', '.mp4', '.avi')) and not f.startswith('._')
    ]
    if mov_files:
        raw_folder = Path(root).name
        label = clean_folder_label(raw_folder)
        if label in TARGET_WORDS:
            for vf in sorted(mov_files):
                video_records.append({
                    "video_path": str(Path(root) / vf),
                    "filename": vf,
                    "raw_folder": raw_folder,
                    "label": label
                })
        else:
            unmatched_folders.add(raw_folder)

df_videos = pd.DataFrame(video_records)
print(f"\nTotal matched videos found: {len(df_videos)}")
if unmatched_folders:
    print(f"Unmatched folders ignored: {unmatched_folders}")

print("\n--- Video Count Per Class ---")
class_counts = df_videos['label'].value_counts().reindex(TARGET_WORDS, fill_value=0)
display(class_counts.to_frame(name="Video Count"))

Checking/Extracting Adjectives_1of8.zip...
Checking/Extracting Greetings_1of2.zip...

Total matched videos found: 209
Unmatched folders ignored: {'Extra'}

--- Video Count Per Class ---


,Video Count
label,
loud,21
quiet,21
happy,21
sad,8
beautiful,8
ugly,8
deaf,8
blind,8
hello,21


In [3]:
NUM_FRAMES = 24  # Evenly spaced frames per video
POSE_INDICES = [11, 12, 13, 14, 15, 16]  # Left/Right Shoulder, Elbow, Wrist

mp_holistic = mp.solutions.holistic

def extract_normalized_sequence(video_path: str, num_frames: int = NUM_FRAMES):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        return None, 0.0

    frame_idxs = np.linspace(0, max(total_frames - 1, 0), num_frames, dtype=int)
    
    # 270 features per frame:
    # 18 (6 pose * 3) + 126 (Left hand: 63 shoulder-rel + 63 wrist-rel) + 126 (Right hand)
    seq = np.full((num_frames, 270), np.nan, dtype=np.float32)
    hand_detected_frames = 0

    with mp_holistic.Holistic(
        static_image_mode=False,
        model_complexity=1,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as holistic:
        for i, f_idx in enumerate(frame_idxs):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(f_idx))
            ret, frame = cap.read()
            if not ret or frame is None:
                continue

            # Downscale 1080p DSLR frames to 640px width for 3x faster MediaPipe inference
            h, w = frame.shape[:2]
            if w > 640:
                scale = 640.0 / w
                frame = cv2.resize(frame, (640, int(h * scale)))

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = holistic.process(rgb)

            # Establish shoulder origin and scale
            if results.pose_landmarks:
                plm = results.pose_landmarks.landmark
                l_sh = np.array([plm[11].x, plm[11].y, plm[11].z], dtype=np.float32)
                r_sh = np.array([plm[12].x, plm[12].y, plm[12].z], dtype=np.float32)
                origin = 0.5 * (l_sh + r_sh)
                sh_width = np.linalg.norm(l_sh[:2] - r_sh[:2])
                if sh_width < 1e-4:
                    sh_width = 1.0

                pose_feats = []
                for idx in POSE_INDICES:
                    pt = np.array([plm[idx].x, plm[idx].y, plm[idx].z], dtype=np.float32)
                    pose_feats.extend((pt - origin) / sh_width)
                seq[i, 0:18] = pose_feats
            else:
                origin = np.array([0.5, 0.5, 0.0], dtype=np.float32)
                sh_width = 1.0

            has_hand = False
            # Left Hand (indices 18 to 144)
            if results.left_hand_landmarks:
                has_hand = True
                lhl = results.left_hand_landmarks.landmark
                pts = np.array([[lm.x, lm.y, lm.z] for lm in lhl], dtype=np.float32)
                wrist = pts[0].copy()
                rel_sh = ((pts - origin) / sh_width).flatten()
                rel_wr = ((pts - wrist) / sh_width).flatten()
                seq[i, 18:81] = rel_sh
                seq[i, 81:144] = rel_wr

            # Right Hand (indices 144 to 270)
            if results.right_hand_landmarks:
                has_hand = True
                rhl = results.right_hand_landmarks.landmark
                pts = np.array([[lm.x, lm.y, lm.z] for lm in rhl], dtype=np.float32)
                wrist = pts[0].copy()
                rel_sh = ((pts - origin) / sh_width).flatten()
                rel_wr = ((pts - wrist) / sh_width).flatten()
                seq[i, 144:207] = rel_sh
                seq[i, 207:270] = rel_wr

            if has_hand:
                hand_detected_frames += 1

    cap.release()
    det_rate = hand_detected_frames / float(num_frames)
    return seq, det_rate

In [4]:
CACHE_FILE = Path("cached_isl_features.joblib")

def sequence_to_feature_row(seq: np.ndarray):
    """
    1. Interpolate missing frames (NaNs) linearly along time axis, then fill remaining with 0.
    2. Compute 5 temporal summary statistics per coordinate:
       mean, std, min, max, and mean absolute frame-to-frame velocity (diff).
    Returns: (1350,) feature vector, number of interpolated values.
    """
    df_seq = pd.DataFrame(seq)
    nan_count_before = int(df_seq.isna().sum().sum())
    
    # Linear interpolation along time + forward/backward fill + 0 fallback
    df_interp = df_seq.interpolate(method='linear', axis=0, limit_direction='both').fillna(0.0)
    arr = df_interp.to_numpy(dtype=np.float32)

    f_mean = np.mean(arr, axis=0)
    f_std  = np.std(arr, axis=0)
    f_min  = np.min(arr, axis=0)
    f_max  = np.max(arr, axis=0)
    f_vel  = np.mean(np.abs(np.diff(arr, axis=0)), axis=0)

    feature_row = np.concatenate([f_mean, f_std, f_min, f_max, f_vel])
    return feature_row, nan_count_before

if CACHE_FILE.exists():
    print(f"Loading cached features from {CACHE_FILE}...")
    cache_data = joblib.load(CACHE_FILE)
    X_raw = cache_data["X_raw"]
    y_raw = cache_data["y_raw"]
    df_meta = cache_data["df_meta"]
    print(f"Loaded X_raw shape: {X_raw.shape}, y_raw shape: {y_raw.shape}")
else:
    features_list = []
    labels_list = []
    meta_records = []

    print("Extracting landmarks from videos (this runs once and caches to disk)...")
    for idx, row in tqdm(df_videos.iterrows(), total=len(df_videos)):
        seq, det_rate = extract_normalized_sequence(row["video_path"])
        if seq is None:
            continue
        feat_vec, nan_count = sequence_to_feature_row(seq)
        features_list.append(feat_vec)
        labels_list.append(row["label"])
        meta_records.append({
            "filename": row["filename"],
            "label": row["label"],
            "hand_detection_rate": det_rate,
            "missing_values_interpolated": nan_count
        })

    X_raw = np.vstack(features_list)
    y_raw = np.array(labels_list)
    df_meta = pd.DataFrame(meta_records)

    joblib.dump({"X_raw": X_raw, "y_raw": y_raw, "df_meta": df_meta}, CACHE_FILE)
    print(f"\nSaved extracted features to {CACHE_FILE}!")

# Quality check report per class
print("\n--- Hand Detection Rate & Missing Value Imputation Summary per Class ---")
quality_summary = df_meta.groupby("label").agg(
    clips=("filename", "count"),
    mean_hand_det_rate=("hand_detection_rate", lambda s: f"{s.mean()*100:.1f}%"),
    min_hand_det_rate=("hand_detection_rate", lambda s: f"{s.min()*100:.1f}%"),
    avg_interpolated_vals=("missing_values_interpolated", "mean")
).reindex(TARGET_WORDS)
display(quality_summary)

# Filter out unusable clips (e.g., hand detected in < 15% of frames)
MIN_DET_RATE = 0.15
keep_mask = (df_meta["hand_detection_rate"] >= MIN_DET_RATE).to_numpy()
print(f"\nClips kept (>= {MIN_DET_RATE*100:.0f}% hand detection): {keep_mask.sum()} / {len(keep_mask)}")
X_clean = X_raw[keep_mask]
y_clean = y_raw[keep_mask]
df_meta_clean = df_meta[keep_mask].reset_index(drop=True)

Extracting landmarks from videos (this runs once and caches to disk)...


  0%|          | 0/209 [00:00<?, ?it/s]


Saved extracted features to cached_isl_features.joblib!

--- Hand Detection Rate & Missing Value Imputation Summary per Class ---


,clips,mean_hand_det_rate,min_hand_det_rate,avg_interpolated_vals
label,,,,
loud,21,85.9%,75.0%,1110.000000
quiet,21,99.6%,91.7%,810.000000
happy,21,80.4%,66.7%,1782.000000
sad,8,100.0%,100.0%,567.000000
beautiful,8,100.0%,100.0%,866.250000
ugly,8,100.0%,100.0%,992.250000
deaf,8,100.0%,100.0%,897.750000
blind,8,100.0%,100.0%,819.000000
hello,21,100.0%,100.0%,648.000000



Clips kept (>= 15% hand detection): 209 / 209


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV, learning_curve
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.pipeline import Pipeline

# Encode string labels to integers 0..12
le = LabelEncoder()
le.fit(TARGET_WORDS)  # Keep consistent order
y_encoded = le.transform(y_clean)
class_names = list(le.classes_)

# Stratified 80/20 Holdout Split
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y_encoded, test_size=0.20, stratify=y_encoded, random_state=42
)

print(f"Training set: {X_train.shape[0]} videos, {X_train.shape[1]} features")
print(f"Holdout Test set: {X_test.shape[0]} videos, {X_test.shape[1]} features")

# Fit StandardScaler + PCA on Training Data ONLY (No Data Leakage)
scaler_vis = StandardScaler()
X_train_scaled = scaler_vis.fit_transform(X_train)
X_test_scaled = scaler_vis.transform(X_test)

pca_full = PCA(random_state=42).fit(X_train_scaled)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)
n_90 = int(np.searchsorted(cum_var, 0.90) + 1)
n_95 = int(np.searchsorted(cum_var, 0.95) + 1)

print(f"Components needed for 90% variance: {n_90} (down from {X_train.shape[1]})")
print(f"Components needed for 95% variance: {n_95} (down from {X_train.shape[1]})")

# Plot 1: PCA Explained Variance & 2D LDA Projection (Practicals 4 & 9)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: PCA Explained Variance
axes[0].plot(range(1, len(cum_var) + 1), cum_var * 100, color='#1f77b4', lw=2.5)
axes[0].axhline(95, color='crimson', linestyle='--', label=f'95% Variance ({n_95} PCs)')
axes[0].axhline(90, color='darkorange', linestyle=':', label=f'90% Variance ({n_90} PCs)')
axes[0].axvline(n_95, color='crimson', linestyle='--')
axes[0].set_title("Module 1: PCA Cumulative Explained Variance (Train Set)", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Number of Principal Components")
axes[0].set_ylabel("Cumulative Explained Variance (%)")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right: PCA(95%) -> 2D LDA Projection
X_train_pca95 = PCA(n_components=0.95, random_state=42).fit_transform(X_train_scaled)
lda_2d = LDA(n_components=2)
X_train_lda2d = lda_2d.fit_transform(X_train_pca95, y_train)

scatter = axes[1].scatter(
    X_train_lda2d[:, 0], X_train_lda2d[:, 1],
    c=y_train, cmap='tab20', s=55, alpha=0.85, edgecolors='k', linewidth=0.4
)
for c_idx, c_name in enumerate(class_names):
    centroid = X_train_lda2d[y_train == c_idx].mean(axis=0)
    axes[1].annotate(c_name, centroid, fontsize=8.5, fontweight='bold',
                     bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.75))
axes[1].set_title("Module 1: 2D Supervised Projection (PCA 95% -> LDA)", fontsize=12, fontweight='bold')
axes[1].set_xlabel("LD 1")
axes[1].set_ylabel("LD 2")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

def get_models():
    return {
        "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
        "k-NN": KNeighborsClassifier(n_neighbors=5, weights='distance'),
        "Decision Tree": DecisionTreeClassifier(max_depth=8, class_weight='balanced', random_state=42),
        "Gaussian Naive Bayes": GaussianNB(),
        "SVM (Linear)": SVC(kernel='linear', C=1.0, probability=True, class_weight='balanced', random_state=42),
        "SVM (RBF)": SVC(kernel='rbf', C=2.0, gamma='scale', probability=True, class_weight='balanced', random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42, n_jobs=-1),
        "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=42),
        "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42),
        "XGBoost": XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, eval_metric='mlogloss', random_state=42, n_jobs=-1),
        "MLP (Neural Net)": MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, early_stopping=False, random_state=42)
    }

def build_dr_steps(dr_name: str):
    if dr_name == "None":
        return [("scaler", StandardScaler())]
    elif dr_name == "PCA_95":
        return [("scaler", StandardScaler()), ("pca", PCA(n_components=0.95, random_state=42))]
    elif dr_name == "PCA_LDA":
        # Regularized PCA(95%) followed by LDA (shrinkage='auto' handles high-D gracefully)
        return [
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=0.95, random_state=42)),
            ("lda", LDA(solver='eigen', shrinkage='auto'))
        ]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']

results_rows = []
print("Running 5-Fold Stratified Cross-Validation across 11 models x 3 DR configs...")

for dr_config in ["None", "PCA_95", "PCA_LDA"]:
    models_dict = get_models()
    for m_name, clf in tqdm(models_dict.items(), desc=f"Config: {dr_config}", leave=False):
        pipe = Pipeline(build_dr_steps(dr_config) + [("clf", clf)])
        try:
            cv_res = cross_validate(
                pipe, X_train, y_train, cv=cv, scoring=scoring,
                return_train_score=True, n_jobs=-1, error_score='raise'
            )
            train_f1 = np.mean(cv_res['train_f1_macro'])
            cv_f1 = np.mean(cv_res['test_f1_macro'])
            cv_f1_std = np.std(cv_res['test_f1_macro'])
            results_rows.append({
                "Model": m_name,
                "DR_Config": dr_config,
                "Train_Macro_F1": round(train_f1, 4),
                "CV_Macro_F1": round(cv_f1, 4),
                "CV_F1_Std": round(cv_f1_std, 4),
                "Overfit_Gap": round(train_f1 - cv_f1, 4),
                "CV_Accuracy": round(np.mean(cv_res['test_accuracy']), 4),
                "CV_Precision": round(np.mean(cv_res['test_precision_macro']), 4),
                "CV_Recall": round(np.mean(cv_res['test_recall_macro']), 4)
            })
        except Exception as e:
            print(f"Skipped {m_name} ({dr_config}): {e}")

df_comparison = pd.DataFrame(results_rows).sort_values(by="CV_Macro_F1", ascending=False).reset_index(drop=True)
print("\n=== TOP 15 MODEL + DIMENSIONALITY REDUCTION COMBINATIONS (BY 5-FOLD CV MACRO F1) ===")
display(df_comparison.head(15))

print("\n=== BEST CONFIG PER MODEL FAMILY ===")
df_best_per_model = df_comparison.sort_values("CV_Macro_F1", ascending=False).drop_duplicates("Model").reset_index(drop=True)
display(df_best_per_model)

In [ ]:
plt.figure(figsize=(14, 6))
x_pos = np.arange(len(df_best_per_model))
width = 0.38

plt.bar(x_pos - width/2, df_best_per_model["Train_Macro_F1"], width, label="Train Macro F1", color="#90caf9", edgecolor="black")
plt.bar(x_pos + width/2, df_best_per_model["CV_Macro_F1"], width, yerr=df_best_per_model["CV_F1_Std"],
        capsize=4, label="5-Fold CV Macro F1", color="#1e88e5", edgecolor="black")

plt.xticks(x_pos, [f"{m}\n({d})" for m, d in zip(df_best_per_model["Model"], df_best_per_model["DR_Config"])], rotation=30, ha="right")
plt.ylabel("Macro F1 Score")
plt.ylim(0, 1.10)
plt.title("Model Comparison: Train vs. 5-Fold CV Macro F1 (Overfitting & Generalization Analysis)", fontsize=13, fontweight="bold")
plt.legend(loc="lower right")
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.ensemble import StackingClassifier

tuning_candidates = {
    "SVM": (
        Pipeline([("scaler", StandardScaler()), ("pca", PCA(random_state=42)), ("clf", SVC(probability=True, random_state=42))]),
        {
            "pca__n_components": [0.90, 0.95, None],
            "clf__kernel": ["linear", "rbf"],
            "clf__C": [0.1, 0.5, 1.0, 2.0, 5.0, 10.0],
            "clf__gamma": ["scale", "auto", 0.001, 0.005],
            "clf__class_weight": ["balanced", None]
        }
    ),
    "Logistic Regression": (
        Pipeline([("scaler", StandardScaler()), ("pca", PCA(random_state=42)), ("clf", LogisticRegression(max_iter=2000, random_state=42))]),
        {
            "pca__n_components": [0.90, 0.95, None],
            "clf__C": [0.05, 0.1, 0.5, 1.0, 5.0],
            "clf__class_weight": ["balanced", None]
        }
    ),
    "Random Forest": (
        Pipeline([("scaler", StandardScaler()), ("pca", PCA(random_state=42)), ("clf", RandomForestClassifier(random_state=42, n_jobs=-1))]),
        {
            "pca__n_components": [0.95, None],
            "clf__n_estimators": [100, 200, 300],
            "clf__max_depth": [None, 8, 12],
            "clf__min_samples_split": [2, 5],
            "clf__class_weight": ["balanced", "balanced_subsample"]
        }
    ),
    "MLP (Neural Net)": (
        Pipeline([("scaler", StandardScaler()), ("pca", PCA(random_state=42)), ("clf", MLPClassifier(max_iter=600, random_state=42))]),
        {
            "pca__n_components": [0.95, None],
            "clf__hidden_layer_sizes": [(128,), (128, 64), (256, 64)],
            "clf__alpha": [0.0001, 0.001, 0.01, 0.1]
        }
    )
}

tuned_summary = []
best_estimators = {}

for name, (pipe, param_grid) in tuning_candidates.items():
    print(f"Tuning {name} via 5-Fold GridSearchCV...")
    gs = GridSearchCV(pipe, param_grid, cv=cv, scoring="f1_macro", n_jobs=-1, refit=True)
    gs.fit(X_train, y_train)
    best_estimators[name] = gs.best_estimator_
    best_idx = gs.best_index_
    std_score = gs.cv_results_["std_test_score"][best_idx]
    tuned_summary.append({
        "Model": name,
        "Best_CV_Macro_F1": round(gs.best_score_, 4),
        "CV_Std": round(std_score, 4),
        "Best_Params": gs.best_params_
    })

# Also evaluate a Stacking Ensemble of the top 3 tuned models (Syllabus Module 3: Stacking)
stack_clf = StackingClassifier(
    estimators=[
        ("svm", best_estimators["SVM"]),
        ("lr", best_estimators["Logistic Regression"]),
        ("rf", best_estimators["Random Forest"])
    ],
    final_estimator=LogisticRegression(class_weight='balanced', random_state=42),
    cv=cv, n_jobs=-1
)
stack_cv = cross_validate(stack_clf, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=-1)
stack_clf.fit(X_train, y_train)
best_estimators["Stacking Ensemble (SVM+LR+RF)"] = stack_clf
tuned_summary.append({
    "Model": "Stacking Ensemble (SVM+LR+RF)",
    "Best_CV_Macro_F1": round(np.mean(stack_cv["test_score"]), 4),
    "CV_Std": round(np.std(stack_cv["test_score"]), 4),
    "Best_Params": {"meta_learner": "LogisticRegression", "base": "Tuned SVM + LR + RF"}
})

df_tuned = pd.DataFrame(tuned_summary).sort_values(by="Best_CV_Macro_F1", ascending=False).reset_index(drop=True)
print("\n=== HYPERPARAMETER TUNING RESULTS (5-FOLD CV) ===")
display(df_tuned)

final_model_name = df_tuned.loc[0, "Model"]
final_model = best_estimators[final_model_name]
print(f"\n>>> SELECTED FINAL MODEL (by highest CV Macro-F1): {final_model_name} <<<")

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, classification_report,
    confusion_matrix, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)

test_acc = accuracy_score(y_test, y_pred)
prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(y_test, y_pred, average='macro', zero_division=0)

print(f"=== FINAL HOLDOUT TEST SET EVALUATION ({final_model_name}) ===")
print(f"Test Accuracy       : {test_acc*100:.2f}%")
print(f"Test Macro Precision: {prec_m*100:.2f}%")
print(f"Test Macro Recall   : {rec_m*100:.2f}%")
print(f"Test Macro F1-Score : {f1_m*100:.2f}%\n")
print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

# Plot Confusion Matrix, OvR ROC Curve, and Bias-Variance Learning Curve
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[0], cbar=False)
axes[0].set_title(f"Holdout Confusion Matrix\n({final_model_name})", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Predicted Word")
axes[0].set_ylabel("True Word")
axes[0].tick_params(axis='x', rotation=45)

# Print Most Confused Word Pairs
cm_off_diag = cm.copy()
np.fill_diagonal(cm_off_diag, 0)
confused_pairs = []
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if cm_off_diag[i, j] > 0:
            confused_pairs.append((class_names[i], class_names[j], cm_off_diag[i, j]))
confused_pairs.sort(key=lambda x: x[2], reverse=True)
print("Most Confused Word Pairs (True -> Predicted : Count):", confused_pairs[:5] if confused_pairs else "None!")

# 2. One-vs-Rest Multi-Class ROC Curve & Macro AUC
y_test_bin = label_binarize(y_test, classes=range(len(class_names)))
fpr_grid = np.linspace(0.0, 1.0, 100)
mean_tpr = np.zeros_like(fpr_grid)
valid_classes = 0

for i, cname in enumerate(class_names):
    if y_test_bin[:, i].sum() > 0:
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
        roc_auc = auc(fpr, tpr)
        mean_tpr += np.interp(fpr_grid, fpr, tpr)
        valid_classes += 1
        axes[1].plot(fpr, tpr, alpha=0.35, lw=1.2, label=f"{cname} (AUC={roc_auc:.2f})")

mean_tpr /= max(valid_classes, 1)
macro_auc = auc(fpr_grid, mean_tpr)
axes[1].plot(fpr_grid, mean_tpr, color='navy', lw=2.8, label=f"Macro-Average ROC (AUC={macro_auc:.3f})")
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_title("One-vs-Rest ROC Curve & Macro AUC", fontsize=12, fontweight='bold')
axes[1].set_xlabel("False Positive Rate (1 - TNR)")
axes[1].set_ylabel("True Positive Rate (Recall)")
axes[1].legend(fontsize=7.5, loc="lower right", ncol=2)
axes[1].grid(alpha=0.3)

# 3. Learning Curve (Bias-Variance Analysis)
train_sizes, train_scores, val_scores = learning_curve(
    final_model, X_train, y_train, cv=cv, scoring='f1_macro',
    train_sizes=np.linspace(0.25, 1.0, 5), n_jobs=-1
)
axes[2].plot(train_sizes, np.mean(train_scores, axis=1), 'o-', color='darkorange', lw=2, label="Training Macro F1")
axes[2].fill_between(train_sizes, np.mean(train_scores, axis=1) - np.std(train_scores, axis=1),
                     np.mean(train_scores, axis=1) + np.std(train_scores, axis=1), alpha=0.15, color='darkorange')
axes[2].plot(train_sizes, np.mean(val_scores, axis=1), 's-', color='forestgreen', lw=2, label="5-Fold CV Macro F1")
axes[2].fill_between(train_sizes, np.mean(val_scores, axis=1) - np.std(val_scores, axis=1),
                     np.mean(val_scores, axis=1) + np.std(val_scores, axis=1), alpha=0.15, color='forestgreen')
axes[2].set_title("Learning Curve (Bias-Variance Trade-off)", fontsize=12, fontweight='bold')
axes[2].set_xlabel("Number of Training Videos")
axes[2].set_ylabel("Macro F1 Score")
axes[2].set_ylim(0.0, 1.05)
axes[2].legend(loc="lower right")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, classification_report,
    confusion_matrix, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)

test_acc = accuracy_score(y_test, y_pred)
prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(y_test, y_pred, average='macro', zero_division=0)

print(f"=== FINAL HOLDOUT TEST SET EVALUATION ({final_model_name}) ===")
print(f"Test Accuracy       : {test_acc*100:.2f}%")
print(f"Test Macro Precision: {prec_m*100:.2f}%")
print(f"Test Macro Recall   : {rec_m*100:.2f}%")
print(f"Test Macro F1-Score : {f1_m*100:.2f}%\n")
print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

# Plot Confusion Matrix, OvR ROC Curve, and Bias-Variance Learning Curve
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[0], cbar=False)
axes[0].set_title(f"Holdout Confusion Matrix\n({final_model_name})", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Predicted Word")
axes[0].set_ylabel("True Word")
axes[0].tick_params(axis='x', rotation=45)

# Print Most Confused Word Pairs
cm_off_diag = cm.copy()
np.fill_diagonal(cm_off_diag, 0)
confused_pairs = []
for i in range(len(class_names)):
    for j in range(len(class_names)):
        if cm_off_diag[i, j] > 0:
            confused_pairs.append((class_names[i], class_names[j], cm_off_diag[i, j]))
confused_pairs.sort(key=lambda x: x[2], reverse=True)
print("Most Confused Word Pairs (True -> Predicted : Count):", confused_pairs[:5] if confused_pairs else "None!")

# 2. One-vs-Rest Multi-Class ROC Curve & Macro AUC
y_test_bin = label_binarize(y_test, classes=range(len(class_names)))
fpr_grid = np.linspace(0.0, 1.0, 100)
mean_tpr = np.zeros_like(fpr_grid)
valid_classes = 0

for i, cname in enumerate(class_names):
    if y_test_bin[:, i].sum() > 0:
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
        roc_auc = auc(fpr, tpr)
        mean_tpr += np.interp(fpr_grid, fpr, tpr)
        valid_classes += 1
        axes[1].plot(fpr, tpr, alpha=0.35, lw=1.2, label=f"{cname} (AUC={roc_auc:.2f})")

mean_tpr /= max(valid_classes, 1)
macro_auc = auc(fpr_grid, mean_tpr)
axes[1].plot(fpr_grid, mean_tpr, color='navy', lw=2.8, label=f"Macro-Average ROC (AUC={macro_auc:.3f})")
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_title("One-vs-Rest ROC Curve & Macro AUC", fontsize=12, fontweight='bold')
axes[1].set_xlabel("False Positive Rate (1 - TNR)")
axes[1].set_ylabel("True Positive Rate (Recall)")
axes[1].legend(fontsize=7.5, loc="lower right", ncol=2)
axes[1].grid(alpha=0.3)

# 3. Learning Curve (Bias-Variance Analysis)
train_sizes, train_scores, val_scores = learning_curve(
    final_model, X_train, y_train, cv=cv, scoring='f1_macro',
    train_sizes=np.linspace(0.25, 1.0, 5), n_jobs=-1
)
axes[2].plot(train_sizes, np.mean(train_scores, axis=1), 'o-', color='darkorange', lw=2, label="Training Macro F1")
axes[2].fill_between(train_sizes, np.mean(train_scores, axis=1) - np.std(train_scores, axis=1),
                     np.mean(train_scores, axis=1) + np.std(train_scores, axis=1), alpha=0.15, color='darkorange')
axes[2].plot(train_sizes, np.mean(val_scores, axis=1), 's-', color='forestgreen', lw=2, label="5-Fold CV Macro F1")
axes[2].fill_between(train_sizes, np.mean(val_scores, axis=1) - np.std(val_scores, axis=1),
                     np.mean(val_scores, axis=1) + np.std(val_scores, axis=1), alpha=0.15, color='forestgreen')
axes[2].set_title("Learning Curve (Bias-Variance Trade-off)", fontsize=12, fontweight='bold')
axes[2].set_xlabel("Number of Training Videos")
axes[2].set_ylabel("Macro F1 Score")
axes[2].set_ylim(0.0, 1.05)
axes[2].legend(loc="lower right")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.neighbors import NearestNeighbors

# --- A. Module 3: L1 (Lasso) vs L2 (Ridge) Regularization & Feature Sparsity ---
lr_l1 = LogisticRegression(solver='saga', l1_ratio=1.0, C=0.5, max_iter=1500, class_weight='balanced', random_state=42)
lr_l2 = LogisticRegression(solver='lbfgs', l1_ratio=0.0, C=0.5, max_iter=1500, class_weight='balanced', random_state=42)

lr_l1.fit(X_train_scaled, y_train)
lr_l2.fit(X_train_scaled, y_train)

l1_zero_pct = np.mean(np.abs(lr_l1.coef_) < 1e-4) * 100
l2_zero_pct = np.mean(np.abs(lr_l2.coef_) < 1e-4) * 100
print(f"[Module 3 Regularization] L1 (Lasso) zeroed out {l1_zero_pct:.1f}% of weights (automatic feature selection) vs {l2_zero_pct:.1f}% for L2 (Ridge).")

# --- B. Module 4: Unsupervised Clustering Tendency (Hopkins Statistic) & Clustering Evaluation ---
def hopkins_statistic(X, m=40, random_state=42):
    rng = np.random.RandomState(random_state)
    n, d = X.shape
    m = min(m, n - 1)
    nbrs = NearestNeighbors(n_neighbors=2).fit(X)
    rand_idx = rng.choice(n, size=m, replace=False)
    u_pts = rng.uniform(np.min(X, axis=0), np.max(X, axis=0), size=(m, d))
    w_dist, _ = nbrs.kneighbors(X[rand_idx], n_neighbors=2)
    u_dist, _ = nbrs.kneighbors(u_pts, n_neighbors=1)
    u_sum1 = np.sum(u_dist[:, 0])
    w_sum1 = np.sum(w_dist[:, 1])
    return u_sum1 / (u_sum1 + w_sum1)

X_dr_cluster = PCA(n_components=12, random_state=42).fit_transform(X_train_scaled)
H_stat = hopkins_statistic(X_dr_cluster)
print(f"[Module 4 Clustering Tendency] Hopkins Statistic H = {H_stat:.3f} (H > 0.5 indicates natural clustering tendency).")

km_labels = KMeans(n_clusters=13, random_state=42, n_init=10).fit_predict(X_dr_cluster)
gmm_labels = GaussianMixture(n_components=13, covariance_type='diag', random_state=42).fit_predict(X_dr_cluster)
mst_labels = AgglomerativeClustering(n_clusters=13, linkage='single').fit_predict(X_dr_cluster)  # Single-linkage = MST clustering
db_labels = DBSCAN(eps=12.0, min_samples=3).fit_predict(X_dr_cluster)

df_clust = pd.DataFrame([
    {"Algorithm": "K-Means (Centroid)", "Silhouette": round(silhouette_score(X_dr_cluster, km_labels), 3), "ARI (vs True Words)": round(adjusted_rand_score(y_train, km_labels), 3)},
    {"Algorithm": "GMM / EM (Model-Based)", "Silhouette": round(silhouette_score(X_dr_cluster, gmm_labels), 3), "ARI (vs True Words)": round(adjusted_rand_score(y_train, gmm_labels), 3)},
    {"Algorithm": "Single-Linkage / MST (Graph-Based)", "Silhouette": round(silhouette_score(X_dr_cluster, mst_labels), 3), "ARI (vs True Words)": round(adjusted_rand_score(y_train, mst_labels), 3)},
    {"Algorithm": "DBSCAN (Density-Based)", "Silhouette": round(silhouette_score(X_dr_cluster, db_labels), 3) if len(set(db_labels)) > 1 else "N/A (1 cluster/noise)", "ARI (vs True Words)": round(adjusted_rand_score(y_train, db_labels), 3)}
])
display(df_clust)

# --- C. Module 5 & 6: Neural Network Optimization (Adam vs SGD Cross-Entropy Loss Curves) ---
mlp_adam = MLPClassifier(hidden_layer_sizes=(128, 64), solver='adam', learning_rate_init=0.001, max_iter=150, random_state=42).fit(X_train_scaled, y_train)
mlp_sgd  = MLPClassifier(hidden_layer_sizes=(128, 64), solver='sgd', learning_rate_init=0.01, momentum=0.9, max_iter=150, random_state=42).fit(X_train_scaled, y_train)

plt.figure(figsize=(8, 4.5))
plt.plot(mlp_adam.loss_curve_, label="MLP with Adam Optimizer (Adaptive Moment)", lw=2.2, color="#d32f2f")
plt.plot(mlp_sgd.loss_curve_, label="MLP with SGD + Momentum", lw=2.2, color="#1976d2", linestyle="--")
plt.title("Module 5 & 6: Cross-Entropy Loss Convergence During MLP Training", fontsize=12, fontweight="bold")
plt.xlabel("Epochs")
plt.ylabel("Cross-Entropy Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
MODEL_BUNDLE_PATH = Path("isl_word_recognizer_final.joblib")
joblib.dump({
    "model": final_model,
    "model_name": final_model_name,
    "label_encoder": le,
    "target_words": TARGET_WORDS
}, MODEL_BUNDLE_PATH)
print(f"Saved complete production model bundle to: {MODEL_BUNDLE_PATH.resolve()}\n")

def predict_isl_video(video_path: str, bundle_path: str = str(MODEL_BUNDLE_PATH)):
    """End-to-end inference function for live Viva demo on any .MOV/.MP4 video."""
    bundle = joblib.load(bundle_path)
    model = bundle["model"]
    le_enc = bundle["label_encoder"]

    seq, det_rate = extract_normalized_sequence(video_path)
    if seq is None:
        return {"error": "Could not read video file."}
    feat_row, _ = sequence_to_feature_row(seq)
    X_single = feat_row.reshape(1, -1)

    pred_idx = model.predict(X_single)[0]
    pred_word = le_enc.inverse_transform([pred_idx])[0]

    proba = model.predict_proba(X_single)[0]
    top3_idxs = np.argsort(proba)[::-1][:3]
    top3 = [(le_enc.inverse_transform([i])[0], round(float(proba[i]) * 100, 2)) for i in top3_idxs]

    print(f"Video File          : {Path(video_path).name}")
    print(f"Hand Detection Rate : {det_rate*100:.1f}%")
    print(f"--> PREDICTED WORD  : '{pred_word.upper()}' ({top3[0][1]:.1f}% confidence)")
    print(f"    Top-3 Candidates: {top3}\n")
    return pred_word, top3

# Test demo on 3 random videos from the dataset
sample_demos = df_videos.sample(3, random_state=7)
for _, r in sample_demos.iterrows():
    print(f"[Ground Truth Folder Label: '{r['label'].upper()}']")
    predict_isl_video(r["video_path"])